# Specialized Types

While Pydantic makes it easy to write custom validators, doing so for common data constraints (like checking if a number is positive, or if a string is a valid email) is reinventing the wheel. Pydantic provides a massive library of **Specialized Types** that come with validation and parsing logic right out of the box.

Broadly, these fall into two categories in the Pydantic documentation:

1. **Standard Pydantic Types:** e.g., `PositiveInt`, `PastDatetime`, `conlist` (constrained lists).
2. **Network Types:** e.g., `HttpUrl`, `EmailStr`, `IPv4Address`. (Network types are especially powerful because they don't just validate strings; they parse them into objects where you can access properties like the `.host` of a URL).

## 1. Using `PositiveInt`

If you need a field to strictly accept an integer greater than zero, you can use `PositiveInt` directly from the Pydantic library.

*Note: Pydantic also provides variants like `NegativeInt`, `NonNegativeInt` (which allows 0), `NonPositiveInt`, and equivalents for floats (e.g., `PositiveFloat`).*

```python
from pydantic import BaseModel, PositiveInt, ValidationError

class Circle(BaseModel):
    # Tuple expecting exactly two integers
    center: tuple[int, int] = (0, 0)
    
    # Strictly greater than 0
    radius: PositiveInt = 1

# ✅ Valid instantiation
c1 = Circle(center=(1, 1), radius=3)
print(c1) # center=(1, 1) radius=3

```

## 2. Validation Error Targeting

When a user inputs invalid data, Pydantic's validation errors are highly descriptive and target the exact index of the failure, even inside nested container types like tuples.

```python
try:
    # ❌ Fails: center contains floats, radius is 0
    c2 = Circle(center=(0.5, 0.5), radius=0)
except ValidationError as e:
    print(e)

```

**The Output:**

```text
3 validation errors for Circle
center.0
  Input should be a valid integer, got a number with a fractional part [type=int_from_float, input_value=0.5, input_type=float]
center.1
  Input should be a valid integer, got a number with a fractional part [type=int_from_float, input_value=0.5, input_type=float]
radius
  Input should be greater than 0 [type=greater_than, input_value=0, input_type=int]

```

Notice how Pydantic uses dot notation (`center.0` and `center.1`) to tell you exactly which element in the tuple failed the integer check.

## 3. Under the Hood: Field Metadata

If you inspect the internal representation of the `radius` field using `Circle.model_fields['radius']`, you will see how Pydantic enforces this.

```python
print(Circle.model_fields['radius'].metadata)
# Output: [Gt(gt=0)]

```

The `PositiveInt` type is actually a shorthand for Python's `Annotated` type, injecting metadata (`Gt(gt=0)` meaning "Greater Than 0") into a standard integer. You will learn how to build your own constrained types using `Annotated` later in the course, but Pydantic's built-in types save you the boilerplate for common constraints.



In [1]:
from pydantic import BaseModel, ValidationError, PositiveInt

In [7]:
class Circle(BaseModel):
    center : tuple[int,int] = (0,0)
    radius : PositiveInt =1
    

In [8]:
Circle.model_fields

{'center': FieldInfo(annotation=tuple[int, int], required=False, default=(0, 0)),
 'radius': FieldInfo(annotation=int, required=False, default=1, metadata=[Gt(gt=0)])}

In [9]:
unit_circle = Circle(radius = 1)

In [10]:
unit_circle

Circle(center=(0, 0), radius=1)

In [11]:
try:
    Circle(center = (0.5,0.5))
except ValidationError as er:
    print(er)

2 validation errors for Circle
center.0
  Input should be a valid integer, got a number with a fractional part [type=int_from_float, input_value=0.5, input_type=float]
    For further information visit https://errors.pydantic.dev/2.13/v/int_from_float
center.1
  Input should be a valid integer, got a number with a fractional part [type=int_from_float, input_value=0.5, input_type=float]
    For further information visit https://errors.pydantic.dev/2.13/v/int_from_float


In [12]:
try:
    Circle(center = (1,1),radius = -1)
except ValidationError as er:
    print(er)

1 validation error for Circle
radius
  Input should be greater than 0 [type=greater_than, input_value=-1, input_type=int]
    For further information visit https://errors.pydantic.dev/2.13/v/greater_than


### 🧠 Interview Preparation: Specialized Types

<details>
<summary><b>1. What is the primary advantage of using Pydantic's specialized types (like `PositiveInt` or `HttpUrl`) over writing your own custom `@field_validator`?</b> (Click to reveal)</summary>
<br>
<b>Answer:</b><br>
There are three main advantages:
1. <b>Less Boilerplate:</b> You don't have to write, test, and maintain custom validation logic for standard constraints.
2. <b>Rich Parsing:</b> Types like `HttpUrl` don't just validate the string; they parse it into an object that exposes attributes like the scheme, host, and path automatically.
3. <b>OpenAPI Schema Generation:</b> Pydantic automatically translates these specialized types into the correct JSON Schema constraints (e.g., `minimum: 1` for a `PositiveInt`), resulting in highly accurate API documentation out of the box. A custom validator is a black box to the schema generator.
</details>

<details>
<summary><b>2. You define a field as `quantity: PositiveInt`. A user submits `0` for the quantity. Does this pass validation? If not, how would you change the type to allow `0`?</b> (Click to reveal)</summary>
<br>
<b>Answer:</b><br>
No, it will fail validation. `PositiveInt` enforces that the value is strictly greater than zero (`> 0`). <br><br>
To allow `0` along with positive numbers, you should use Pydantic's `NonNegativeInt` type, which enforces that the value is greater than or equal to zero (`>= 0`).
</details>

<details>
<summary><b>3. You have a Pydantic model with a field `coordinates: tuple[int, int]`. A user submits `{"coordinates": [10, 5.5]}`. How does Pydantic format the resulting validation error to help the developer pinpoint the issue?</b> (Click to reveal)</summary>
<br>
<b>Answer:</b><br>
Pydantic uses dot-notation indexing for container types. Instead of just failing the entire `coordinates` field, it will specifically flag `coordinates.1` (the second item in the tuple) and state that it expected an integer but received a number with a fractional part.
</details>
